# Assignment 2 — KNN and Classification Tree


In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier


In [26]:
# Settings
RANDOM_STATE = 42
TARGET = "attack_cat"
ID_COLUMN = "id"
CATEGORICAL_FEATURES = ["proto", "service", "state"]

RUN_EDA = True
RUN_BASELINE_CV = True
RUN_KNN_TUNING = True
RUN_TREE_TUNING = True
RUN_TREE_REFINED = True
RUN_FINAL_COMPARISON = True


In [3]:
# File paths
DATA_PATH = Path("networkTraffic.csv")
FEATURES_PATH = Path("features.csv")
ATTACK_MAP_PATH = Path("attack_category_map.csv")

OUTPUT_DIR = Path("assignment2_outputs")
MODEL_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# Load data
df = pd.read_csv(DATA_PATH)
features = pd.read_csv(FEATURES_PATH)
attack_map = pd.read_csv(ATTACK_MAP_PATH)

print("Network traffic shape:", df.shape)
print("Features shape:", features.shape)
print("Attack map shape:", attack_map.shape)

print("\nColumns:")
print(df.columns.tolist())


Network traffic shape: (257673, 44)
Features shape: (42, 3)
Attack map shape: (10, 2)

Columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat']


In [8]:
# Replace ? with NaN
df = df.replace("?", np.nan).copy()

# Check missing values
missing_counts = df.isna().sum()

print("\nMissing values:")
print(missing_counts[missing_counts > 0])

# Set feature types
numerical_features = [
    col for col in df.columns
    if col not in CATEGORICAL_FEATURES + [TARGET, ID_COLUMN]
]

for col in numerical_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check target
if df[TARGET].isna().sum() > 0:
    raise ValueError("Target contains missing values.")

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise"
).astype(int)

# Check numerical missing values
numeric_missing = df[numerical_features].isna().sum()
numeric_missing = numeric_missing[numeric_missing > 0]

if len(numeric_missing) > 0:
    raise ValueError(
        "Unexpected missing values in numerical features:\n"
        + numeric_missing.to_string()
    )


Missing values:
service    141321
dtype: int64


In [12]:
# EDA
if RUN_EDA:

    print("\nData types:")
    print(df.dtypes)

    missing_count = df.isna().sum()
    missing_percentage = missing_count / len(df) * 100

    missing_summary = pd.DataFrame({
        "Count": missing_count,
        "Percentage": missing_percentage
    })

    missing_summary = missing_summary[
        missing_summary["Count"] > 0
    ].sort_values("Count", ascending=False)

    print("\nMissing values:")
    print(missing_summary)

    print("\nUnique id values:", df[ID_COLUMN].nunique())
    print("Number of observations:", len(df))
    print("Duplicate rows:", df.duplicated().sum())

    class_counts = (
        df[TARGET]
        .value_counts()
        .sort_index()
        .rename_axis("Mapping")
        .reset_index(name="Count")
    )

    class_counts["Percentage"] = (
        class_counts["Count"] / len(df) * 100
    )

    attack_map["Mapping"] = pd.to_numeric(
        attack_map["Mapping"],
        errors="raise"
    )

    target_summary = class_counts.merge(
        attack_map[["Mapping", "Attack Category Name"]],
        on="Mapping",
        how="left"
    )

    target_summary = target_summary[
        ["Mapping", "Attack Category Name", "Count", "Percentage"]
    ]

    print("\nTarget class distribution:")
    print(target_summary.to_string(index=False))

    numeric_summary = df[numerical_features].describe().T
    numeric_summary = numeric_summary[
        ["min", "max", "mean", "std", "25%", "50%", "75%"]
    ]

    print("\nNumerical feature summary:")
    print(numeric_summary.to_string())

    correlation_matrix = df[numerical_features].corr()

    upper_triangle = correlation_matrix.where(
        np.triu(
            np.ones(correlation_matrix.shape),
            k=1
        ).astype(bool)
    )

    correlation_pairs = (
        upper_triangle
        .stack()
        .rename("Correlation")
        .reset_index()
        .rename(columns={
            "level_0": "Feature_1",
            "level_1": "Feature_2"
        })
    )

    correlation_pairs["Absolute_Correlation"] = (
        correlation_pairs["Correlation"].abs()
    )

    correlation_pairs = correlation_pairs.sort_values(
        "Absolute_Correlation",
        ascending=False
    )

    print("\nStrongest correlations:")
    print(correlation_pairs.head(20).to_string(index=False))

    Q1 = df[numerical_features].quantile(0.25)
    Q3 = df[numerical_features].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_counts = (
        (df[numerical_features] < lower_bound) |
        (df[numerical_features] > upper_bound)
    ).sum()

    outlier_summary = pd.DataFrame({
        "Count": outlier_counts,
        "Percentage": outlier_counts / len(df) * 100
    }).sort_values("Percentage", ascending=False)

    print("\nOutliers using IQR rule:")
    print(outlier_summary.to_string())



Data types:
id                     int64
dur                  float64
proto                 object
service               object
state                 object
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
c

In [13]:
# Predictors and target
X = df.drop(columns=[TARGET, ID_COLUMN]).copy()
y = df[TARGET].copy()

categorical_features = CATEGORICAL_FEATURES.copy()

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("\nX shape:", X.shape)
print("y shape:", y.shape)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

# KNN preprocessing
knn_preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Tree preprocessing
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)



X shape: (257673, 42)
y shape: (257673,)


In [14]:
# Models
knn_model = Pipeline(
    steps=[
        ("preprocessing", knn_preprocessor),
        (
            "knn",
            KNeighborsClassifier(
                n_jobs=-1
            )
        )
    ]
)

tree_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessor),
        (
            "tree",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted"
}


In [17]:
# Baseline cross-validation
if RUN_BASELINE_CV:

    print("\nKNN baseline")

    knn_baseline = cross_validate(
        knn_model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    print("Mean accuracy:", knn_baseline["test_accuracy"].mean())
    print("Mean macro F1:", knn_baseline["test_macro_f1"].mean())
    print("Mean weighted F1:", knn_baseline["test_weighted_f1"].mean())
    print("Macro F1 std:", knn_baseline["test_macro_f1"].std(ddof=0))

    print("\nClassification Tree baseline")

    tree_baseline = cross_validate(
        tree_model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    print("Mean accuracy:", tree_baseline["test_accuracy"].mean())
    print("Mean macro F1:", tree_baseline["test_macro_f1"].mean())
    print("Mean weighted F1:", tree_baseline["test_weighted_f1"].mean())
    print("Macro F1 std:", tree_baseline["test_macro_f1"].std(ddof=0))



KNN baseline
Mean accuracy: 0.7894424283135066
Mean macro F1: 0.5035685223083388
Mean weighted F1: 0.7844480050558794
Macro F1 std: 0.006077484149283832

Classification Tree baseline
Mean accuracy: 0.8086799941602634
Mean macro F1: 0.5809251916118685
Mean weighted F1: 0.8053793479507965
Macro F1 std: 0.003637684822292047


In [18]:
baseline_results = pd.DataFrame({
    "Model": [
        "KNN",
        "Classification Tree"
    ],
    "Mean_Accuracy": [
        knn_baseline["test_accuracy"].mean(),
        tree_baseline["test_accuracy"].mean()
    ],
    "Mean_Macro_F1": [
        knn_baseline["test_macro_f1"].mean(),
        tree_baseline["test_macro_f1"].mean()
    ],
    "Std_Macro_F1": [
        knn_baseline["test_macro_f1"].std(ddof=0),
        tree_baseline["test_macro_f1"].std(ddof=0)
    ],
    "Mean_Weighted_F1": [
        knn_baseline["test_weighted_f1"].mean(),
        tree_baseline["test_weighted_f1"].mean()
    ]
})

baseline_results.to_csv(
    OUTPUT_DIR / "baseline_model_comparison.csv",
    index=False
)

baseline_results

,Model,Mean_Accuracy,Mean_Macro_F1,Std_Macro_F1,Mean_Weighted_F1
0,KNN,0.789442,0.503569,0.006077,0.784448
1,Classification Tree,0.808680,0.580925,0.003638,0.805379


## KNN tuning

In [25]:
# Final KNN tuning
if RUN_KNN_TUNING:

    knn_parameters = {
        "knn__n_neighbors": [3, 4, 5, 6, 7],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["euclidean", "manhattan"]
    }

    print("\nRunning KNN GridSearchCV...")

    knn_search = GridSearchCV(
        estimator=knn_model,
        param_grid=knn_parameters,
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=2,
        return_train_score=False,
        refit=True
    )

    knn_search.fit(X, y)

    print("\nBest KNN parameters:")
    print(knn_search.best_params_)

    print("\nBest KNN macro F1:")
    print(knn_search.best_score_)

    knn_results = pd.DataFrame(
        knn_search.cv_results_
    ).sort_values("rank_test_score")

    knn_columns = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_knn__n_neighbors",
        "param_knn__weights",
        "param_knn__metric"
    ]

    print("\nTop KNN configurations:")
    print(
        knn_results[knn_columns]
        .head(20)
        .to_string(index=False)
    )

    knn_results.to_csv(
        OUTPUT_DIR / "knn_tuning_results.csv",
        index=False
    )

    joblib.dump(
        knn_search.best_estimator_,
        MODEL_DIR / "best_knn_model.pkl"
    )


Running KNN GridSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform; total time= 8.3min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform; total time= 8.5min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform; total time= 8.5min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform; total time= 8.3min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform; total time= 8.4min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=distance; total time= 8.4min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=distance; total time= 8.0min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=distance; total time= 8.0min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=distance; total time= 8.2min
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=dis

## Classification Tree tuning

In [20]:
# Broad tree tuning
if RUN_TREE_TUNING:

    tree_parameters = {
        "tree__criterion": ["gini", "entropy"],
        "tree__max_depth": [None, 10, 20, 30, 40],
        "tree__min_samples_split": [2, 5, 10, 20],
        "tree__min_samples_leaf": [1, 2, 5, 10, 20],
        "tree__class_weight": [None, "balanced"]
    }

    print("\nRunning tree RandomizedSearchCV...")

    tree_search = RandomizedSearchCV(
        estimator=tree_model,
        param_distributions=tree_parameters,
        n_iter=30,
        scoring="f1_macro",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2,
        return_train_score=True,
        refit=True
    )

    tree_search.fit(X, y)

    print("\nBest tree parameters:")
    print(tree_search.best_params_)

    print("\nBest tree macro F1:")
    print(tree_search.best_score_)

    tree_results = pd.DataFrame(
        tree_search.cv_results_
    ).sort_values("rank_test_score")

    tree_columns = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_tree__criterion",
        "param_tree__max_depth",
        "param_tree__min_samples_split",
        "param_tree__min_samples_leaf",
        "param_tree__class_weight"
    ]

    print("\nTop tree configurations:")
    print(
        tree_results[tree_columns]
        .head(15)
        .to_string(index=False)
    )

    tree_results.to_csv(
        OUTPUT_DIR / "tree_tuning_results.csv",
        index=False
    )

    joblib.dump(
        tree_search.best_estimator_,
        MODEL_DIR / "best_tree_broad.pkl"
    )

    with open(
        OUTPUT_DIR / "best_tree_parameters.json",
        "w"
    ) as file:
        json.dump(
            tree_search.best_params_,
            file,
            indent=4
        )



Running tree RandomizedSearchCV...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best tree parameters:
{'tree__min_samples_split': 5, 'tree__min_samples_leaf': 5, 'tree__max_depth': None, 'tree__criterion': 'gini', 'tree__class_weight': None}

Best tree macro F1:
0.5870925135450806

Top tree configurations:
 rank_test_score  mean_test_score  std_test_score param_tree__criterion param_tree__max_depth  param_tree__min_samples_split  param_tree__min_samples_leaf param_tree__class_weight
               1         0.587093        0.003198                  gini                  None                              5                             5                     None
               2         0.580977        0.008981               entropy                  None                             10                            10                     None
               3         0.580925        0.003638                  gini                  None                              2         

In [23]:
# Final refined tree tuning
if RUN_TREE_REFINED:

    refined_tree_model = Pipeline(
        steps=[
            (
                "preprocessing",
                tree_preprocessor
            ),
            (
                "tree",
                DecisionTreeClassifier(
                    criterion="gini",
                    max_depth=None,
                    class_weight=None,
                    random_state=RANDOM_STATE
                )
            )
        ]
    )

    refined_parameters = {
        "tree__min_samples_leaf": [4, 5, 6],
        "tree__min_samples_split": [8, 10, 12, 15]
    }

    print("\nRunning final tree refinement...")

    refined_tree_search = GridSearchCV(
        estimator=refined_tree_model,
        param_grid=refined_parameters,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=2,
        return_train_score=True,
        refit=True
    )

    refined_tree_search.fit(X, y)

    print("\nBest final tree parameters:")
    print(refined_tree_search.best_params_)

    print("\nBest final tree macro F1:")
    print(refined_tree_search.best_score_)

    refined_results = pd.DataFrame(
        refined_tree_search.cv_results_
    ).sort_values("rank_test_score")

    print("\nFinal tree configurations:")
    print(
        refined_results[
            [
                "rank_test_score",
                "mean_test_score",
                "std_test_score",
                "param_tree__min_samples_leaf",
                "param_tree__min_samples_split"
            ]
        ].to_string(index=False)
    )

    refined_results.to_csv(
        OUTPUT_DIR / "tree_final_refinement.csv",
        index=False
    )

    joblib.dump(
        refined_tree_search.best_estimator_,
        MODEL_DIR / "best_tree_model.pkl"
    )


Running final tree refinement...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best final tree parameters:
{'tree__min_samples_leaf': 4, 'tree__min_samples_split': 10}

Best final tree macro F1:
0.588710856674779

Final tree configurations:
 rank_test_score  mean_test_score  std_test_score  param_tree__min_samples_leaf  param_tree__min_samples_split
               1         0.588711        0.004829                             4                             10
               2         0.588448        0.004403                             4                             12
               3         0.588420        0.005025                             4                             15
               4         0.587093        0.003198                             5                              8
               4         0.587093        0.003198                             5                             10
               6         0.586579        0.005232                           

## Final comparison

In [ ]:
# Final model comparison
if RUN_FINAL_COMPARISON:

    knn_path = MODEL_DIR / "best_knn_model.pkl"
    tree_path = MODEL_DIR / "best_tree_model.pkl"

    if not knn_path.exists():
        raise FileNotFoundError(
            "Run KNN tuning first."
        )

    if not tree_path.exists():
        raise FileNotFoundError(
            "Run tree tuning first."
        )

    best_knn = joblib.load(knn_path)
    best_tree = joblib.load(tree_path)

    labels = sorted(y.unique())

    attack_map["Mapping"] = pd.to_numeric(
        attack_map["Mapping"],
        errors="raise"
    )

    label_to_name = dict(
        zip(
            attack_map["Mapping"],
            attack_map["Attack Category Name"]
        )
    )

    class_names = [
        str(label_to_name.get(label, label))
        for label in labels
    ]

    models = {
        "KNN": best_knn,
        "Classification Tree": best_tree
    }

    final_results = []

    for model_name, model in models.items():

        print(f"\n{model_name}")

        cv_jobs = 1 if model_name == "KNN" else -1

        result = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring,
            n_jobs=cv_jobs
        )

        predictions = cross_val_predict(
            model,
            X,
            y,
            cv=cv,
            n_jobs=cv_jobs
        )

        mean_accuracy = result["test_accuracy"].mean()
        mean_macro_f1 = result["test_macro_f1"].mean()
        macro_f1_std = result["test_macro_f1"].std(ddof=0)
        mean_weighted_f1 = result["test_weighted_f1"].mean()

        print("Mean accuracy:", mean_accuracy)
        print("Mean macro F1:", mean_macro_f1)
        print("Macro F1 std:", macro_f1_std)
        print("Mean weighted F1:", mean_weighted_f1)

        print("\nClassification report:")
        print(
            classification_report(
                y,
                predictions,
                labels=labels,
                target_names=class_names,
                zero_division=0
            )
        )

        safe_name = (
            model_name
            .lower()
            .replace(" ", "_")
        )

        report = classification_report(
            y,
            predictions,
            labels=labels,
            target_names=class_names,
            zero_division=0,
            output_dict=True
        )

        pd.DataFrame(report).T.to_csv(
            OUTPUT_DIR /
            f"{safe_name}_classification_report.csv"
        )

        fig, ax = plt.subplots(figsize=(10, 8))

        ConfusionMatrixDisplay.from_predictions(
            y,
            predictions,
            labels=labels,
            display_labels=class_names,
            normalize="true",
            values_format=".2f",
            xticks_rotation=45,
            ax=ax
        )

        ax.set_title(
            f"{model_name} - Normalized Confusion Matrix"
        )

        fig.tight_layout()

        fig.savefig(
            OUTPUT_DIR /
            f"{safe_name}_confusion_matrix.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

        final_results.append({
            "Model": model_name,
            "Mean_Accuracy": mean_accuracy,
            "Mean_Macro_F1": mean_macro_f1,
            "Std_Macro_F1": macro_f1_std,
            "Mean_Weighted_F1": mean_weighted_f1
        })

    final_results = pd.DataFrame(final_results)

    print("\nFinal comparison:")
    print(final_results.to_string(index=False))

    final_results.to_csv(
        OUTPUT_DIR / "final_model_comparison.csv",
        index=False
    )



KNN
